# ДЗ №3. Задание 1
### Выполнил: Лёвин В.А.
### Группа: ИУ6-54Б

Количество сообщений.


## Подготовка

Сначала запускаем Kafka по инструкции в README. Сборщик `telegram_producer.py` запускается отдельно и передаёт новые сообщения восьми каналов в Kafka.

Время окон — время поступления сообщений в Spark. Окна длиной 1 и 10 минут сдвигаются на 30 секунд. Сразу после запуска они заполнены только частично.

In [6]:
import json
import socket
import sys
from operator import add
from threading import Event, Thread

from kafka import KafkaConsumer
from pyspark import SparkConf, SparkContext
from pyspark.streaming import StreamingContext

conf = SparkConf().setMaster("local[2]").setAppName("HW3-Messages")
sc = SparkContext.getOrCreate(conf=conf)
sc.pythonExec = sys.executable
ssc = StreamingContext(sc, 30)

## 1. Поток из Kafka

Читаем только новые сообщения. Запускаем сборщик Telegram после запуска последней ячейки этой тетради.

In [7]:
consumer = KafkaConsumer(
    "telegram_messages", bootstrap_servers="localhost:9092",
    auto_offset_reset="latest", enable_auto_commit=False,
)
server = socket.socket()
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server.bind(("127.0.0.1", 9999))
server.listen(1)
stopped = Event()


def forward_kafka():
    try:
        connection, _ = server.accept()
        with connection:
            while not stopped.is_set():
                records = consumer.poll(timeout_ms=1000)
                for batch in records.values():
                    for record in batch:
                        connection.sendall(record.value + b"\n")
    except (BrokenPipeError, ConnectionResetError):
        pass
    finally:
        consumer.close()
        server.close()


worker = Thread(target=forward_kafka, daemon=True)
lines = ssc.socketTextStream("127.0.0.1", 9999)

ERROR:kafka.conn:Connect attempt to <BrokerConnection node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv6 ('::1', 9092, 0, 0)]> returned error 61. Disconnecting.
ERROR:kafka.conn:Connect attempt to <BrokerConnection node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv6 ('::1', 9092, 0, 0)]> returned error 61. Disconnecting.


## 2. Подсчёт сообщений в окнах

Сначала считаем сообщения каждого отправителя за 30 секунд, затем суммируем эти количества в окнах 60 и 600 секунд. Для каждого окна выводим всех отправителей по убыванию количества сообщений.

Если у отправителя нет username, сборщик передаёт его ID. В анонимных публикациях Telegram может указывать сам канал вместо автора.

In [8]:
def parse_message(line):
    message = json.loads(line)
    return message["username"], 1


def message_count(item):
    return item[1]


def sort_counts(rdd):
    return rdd.sortBy(message_count, ascending=False, numPartitions=1)


def show_minute(time, rdd):
    print(f"\n{time} — окно 1 минута")
    for username, count in rdd.collect():
        print(f"{username}\t{count}")


def show_ten_minutes(time, rdd):
    print(f"\n{time} — окно 10 минут")
    for username, count in rdd.collect():
        print(f"{username}\t{count}")


batch_counts = lines.map(parse_message).reduceByKey(add)
minute_counts = batch_counts.window(60, 30).reduceByKey(add).transform(sort_counts)
ten_minute_counts = batch_counts.window(600, 30).reduceByKey(add).transform(sort_counts)
minute_counts.foreachRDD(show_minute)
ten_minute_counts.foreachRDD(show_ten_minutes)

## 3. Запуск

Ячейка работает, пока запущен поток. Каждые 30 секунд появляются две таблицы: за последнюю минуту и за последние 10 минут. Пустое окно содержит только заголовок.

После запуска ячейки запускаем сборщик Telegram в терминале. Для остановки потока нажимаем Interrupt в панели тетради, сборщик останавливаем через Ctrl+C. Чтобы повторить запуск всех ячеек, сначала перезапускаем ядро.

In [ ]:
worker.start()
ssc.start()
try:
    ssc.awaitTermination()
except KeyboardInterrupt:
    pass
finally:
    stopped.set()
    ssc.stop(stopSparkContext=False, stopGraceFully=True)

ERROR:kafka.conn:Connect attempt to <BrokerConnection node_id=1 host=localhost:9092 <connecting> [IPv6 ('::1', 9092, 0, 0)]> returned error 61. Disconnecting.



2026-11-25 11:29:30 — окно 1 минута

2026-11-25 11:29:30 — окно 10 минут


ERROR:root:KeyboardInterrupt while sending command.                 (0 + 1) / 1]
Traceback (most recent call last):
  File "/Users/mac/Documents/Big Data/DZ_3/.venv/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mac/Documents/Big Data/DZ_3/.venv/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mac/.local/share/uv/python/cpython-3.11.14-macos-aarch64-none/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
26/11/25 11:29:48 ERROR ReceiverTracker: Deregistered receiver for stream 0: Stopped by driver
26/11/25 11:29:48 WARN SocketReceiver: Error receiving data
java.net.SocketException: Socket closed
	at java.base/sun.ni


2026-11-25 11:30:30 — окно 1 минута

2026-11-25 11:30:30 — окно 10 минут


26/11/25 12:07:11 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 830078 ms exceeds timeout 120000 ms
26/11/25 12:07:11 WARN SparkContext: Killing executors is not supported by current scheduler.
26/11/25 12:07:16 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1240)
	at o